# 03 - Pandas Analysis

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

pd.set_option("display.float_format", '{:,.2f}'.format)
pd.set_option("display.max_rows", 50)

engine = create_engine("sqlite:///data/movies.db")

movies = pd.read_sql("SELECT * FROM movies",engine,parse_dates=["release_date"])
genres = pd.read_sql("SELECT * FROM genres",engine)
movie_genres = pd.read_sql("SELECT * FROM movie_genres",engine)
cast = pd.read_sql("SELECT * FROM cast",engine)
crew = pd.read_sql("SELECT * FROM crew",engine)
movie_keywords = pd.read_sql("SELECT * FROM movie_keywords",engine)

In [3]:
movies.shape

(2503, 12)

## Unique movies, genres, cast members and crew members

In [4]:
counts = pd.Series({
    "movies": movies["movie_id"].nunique(),
    "genres": genres["genre_id"].nunique(),
    "cast_members" : cast["person_id"].nunique(),
    "crew_members": crew["person_id"].nunique()
})
print(counts)

movies           2503
genres             19
cast_members    11300
crew_members     2848
dtype: int64


### Insight

The dataset contains 2,503 unique movies, 19 genres, 11,300 unique cast members and 2,848 unique crew members. Each count uses nunique() on the ID column: movie_id, genre_id and person_id (for both cast and crew). person_id is used instead of the row IDs, because the row IDs only number the rows, and a person who appears in several movies has several rows but is still one person.

The cast table has 24,902 rows but only 11,300 unique people, so many actors appear in more than one movie.

## Missing or zero budget/revenue, and how to treat them

In [5]:
zero_finding = movies[(movies["budget"]==0) | (movies["revenue"]==0)]
print("movie with 0 budget or 0 revenue: ", len(zero_finding))
print("movies with 0 budget: ", (movies["budget"]==0).sum())
print("movies with 0 revenue: ",(movies["revenue"]==0).sum())
print("movies with both budget and revenue 0: ", ((movies["budget"]==0)& (movies["revenue"]==0)).sum())
zero_finding[["title","budget","revenue"]].head()

movie with 0 budget or 0 revenue:  133
movies with 0 budget:  90
movies with 0 revenue:  102
movies with both budget and revenue 0:  59


,title,budget,revenue
324,Zack Snyder's Justice League,"70,000,000.00",0.00
340,Bird Box,"19,800,000.00",0.00
403,365 Days,0.00,"9,458,590.00"
468,To All the Boys I've Loved Before,0.00,0.00
554,Prey,"65,000,000.00",0.00


In [6]:
movies_clean = movies.copy()
movies_clean[["budget","revenue"]] = movies_clean[["budget","revenue"]].replace(0,np.nan)
print("0 budget movies in new DF: ",(movies_clean["budget"]==0).sum())
print("0 revenue movies in new DF: ",(movies_clean["revenue"]==0).sum())
print("0 budget movies in original DF: ",(movies["budget"]==0).sum())
print("0 revenue movies in original DF: ",(movies["revenue"]==0).sum())
print("******************")
print("budget & revenue null values count in original DF: ")
print(movies[["budget","revenue"]].isnull().sum())
print("budget & revenue null values count in new DF: ")
print(movies_clean[["budget","revenue"]].isnull().sum())

0 budget movies in new DF:  0
0 revenue movies in new DF:  0
0 budget movies in original DF:  90
0 revenue movies in original DF:  102
******************
budget & revenue null values count in original DF: 
budget     0
revenue    0
dtype: int64
budget & revenue null values count in new DF: 
budget      90
revenue    102
dtype: int64


### Insight

1. 133 movies have a zero budget or a zero revenue. Of these, 90 have a zero budget, 102 have a zero revenue, and 59 have both. (90 + 102 - 59 = 133.)

2. In TMDB, 0 means the value is unknown, not that the movie was free or earned nothing. For example, Zack Snyder's Justice League and Bird Box have a budget but zero revenue (possibly because they were streaming releases), 365 Days has revenue but zero budget, and To All the Boys I've Loved Before has both zero.

Note: the OR condition (|) is used to include movies where either value is zero, and the AND condition (&) is used to find movies where both are zero.

The zeros in budget and revenue were replaced with NaN in a copy of the data (movies_clean), and the original movies table was left unchanged. movies_clean is used for profit, ROI and other financial calculations, so unknown values do not distort the results. The original movies table is used for ratings, popularity and runtime, so these movies are not lost.

## Profit and ROI for movies with valid budget and revenue

In [7]:
valid_budget_and_revenue_movies = movies_clean.dropna(subset=["revenue","budget"]).copy()
valid_budget_and_revenue_movies["profit"] = valid_budget_and_revenue_movies["revenue"]-valid_budget_and_revenue_movies["budget"]
valid_budget_and_revenue_movies["ROI"] = valid_budget_and_revenue_movies["profit"]/valid_budget_and_revenue_movies["budget"]*100
valid_budget_and_revenue_movies[["title","revenue","budget","profit","ROI"]]

,title,revenue,budget,profit,ROI
0,Interstellar,"746,606,706.00","165,000,000.00","581,606,706.00",352.49
1,Inception,"839,030,630.00","160,000,000.00","679,030,630.00",424.39
2,The Avengers,"1,518,815,515.00","220,000,000.00","1,298,815,515.00",590.37
3,The Dark Knight,"1,004,558,444.00","185,000,000.00","819,558,444.00",443.00
4,Avatar,"2,923,706,026.00","237,000,000.00","2,686,706,026.00","1,133.63"
...,...,...,...,...,...
2498,Daybreakers,"51,417,188.00","20,000,000.00","31,417,188.00",157.09
2499,Date Night,"87,595,677.74","93,214,796.42","-5,619,118.67",-6.03
2500,Transformers,"274,746,695.42","110,098,767.08","164,647,928.34",149.55
2501,The SpongeBob Movie: Sponge Out of Water,"570,657,513.93","39,284,878.87","531,372,635.07","1,352.61"


In [8]:
valid_budget_and_revenue_movies.shape

(2370, 14)

### Insight

1. 2,370 movies have both a known budget and a known revenue, so only these are used for profit and ROI. Movies with a missing (NaN) budget or revenue were dropped using dropna(subset=["revenue","budget"]) on movies_clean.

2. Profit = revenue - budget, and ROI = profit / budget x 100 (in percent). For example, Interstellar has an ROI of about 352%, Inception about 424% and Avatar about 1,133%.

3. Some of the movies have a negative profit, meaning their revenue was lower than their budget. For example, Date Night.

Note: ROI is calculated from box-office revenue and production budget only. Marketing and other costs are not included, so it is not the true return.

Recommendation: Profit and ROI should be calculated only for movies with known figures, because including unknown (zero) values would give misleading results.

## Top 15 movies by ROI (budget of at least $1 million)

In [9]:
valid_movies_with_budget_above_1million = valid_budget_and_revenue_movies[valid_budget_and_revenue_movies["budget"]>=1000000]
valid_movies_with_budget_above_1million = valid_movies_with_budget_above_1million.sort_values(by="ROI", ascending=False)
valid_movies_with_budget_above_1million[["title","revenue","budget","profit","ROI"]].head(15)

,title,revenue,budget,profit,ROI
1767,Dragon Ball Super: Broly,"125,002,821.00","1,000,000.00","124,002,821.00","12,400.28"
535,Snow White and the Seven Dwarfs,"184,925,486.00","1,488,423.00","183,437,063.00","12,324.26"
1791,The Rocky Horror Picture Show,"171,181,400.00","1,400,000.00","169,781,400.00","12,127.24"
467,Rocky,"117,253,345.00","1,000,000.00","116,253,345.00","11,625.33"
1227,Gone with the Wind,"402,352,579.00","4,000,000.00","398,352,579.00","9,958.81"
716,The Jungle Book,"378,000,000.00","4,000,000.00","374,000,000.00","9,350.00"
635,Cinderella,"263,600,000.00","2,900,000.00","260,700,000.00","8,989.66"
372,Saw,"104,045,735.00","1,200,000.00","102,845,735.00","8,570.48"
719,One Hundred and One Dalmatians,"303,000,000.00","3,600,000.00","299,400,000.00","8,316.67"
263,E.T. the Extra-Terrestrial,"797,307,407.00","10,500,000.00","786,807,407.00","7,493.40"


### Insight

1. From the movies with a known budget and revenue, only those with a budget of at least $1 million were kept.

2. The movies were sorted by ROI from highest to lowest, and the top 15 are listed.

3. Dragon Ball Super: Broly has the highest ROI (about 12,400%), which means its profit was about 124 times its budget. Snow White and the Seven Dwarfs (about 12,324%) and The Rocky Horror Picture Show (about 12,127%) follow closely.

4. The top 15 also include Rocky, Gone with the Wind, The Jungle Book, Cinderella, Saw, E.T., Star Wars, Jaws and Grease. Many of them are older films, and their budgets are not adjusted for inflation, so their ROI looks very high.

Recommendation: Low-budget films with a strong audience can give a very high return, so ROI is a useful measure of efficiency.

## Movies released per year, and the peak year

In [10]:
movies["year"] = movies["release_date"].dt.year
print("number of movies released per year: ")
print(movies["year"].value_counts())
print("Highest movie released year: ", movies["year"].value_counts().idxmax())
print("Number of movies in the highest movie released year: ", movies["year"].value_counts().max())

number of movies released per year: 
year
2016    121
2013    110
2017    109
2014    105
2018    102
       ... 
1952      1
1927      1
1969      1
1922      1
1921      1
Name: count, Length: 89, dtype: int64
Highest movie released year:  2016
Number of movies in the highest movie released year:  121


### Insight

1. The year was extracted from release_date and stored in a new column. value_counts() was then used to count the movies released in each year.

2. 2016 is the peak year, with 121 movies, followed by 2013 (110 movies) and 2017 (109 movies). idxmax() gives the peak year and max() gives its count.

3. Most movies in the dataset are from the 2000s and 2010s, while older years have only a few movies each.

4. 2025 (26 movies) and 2026 (8 movies) have far fewer movies than recent years, so these years are probably incomplete in the dataset.

Recommendation: The dataset has more movies from recent years, so trends from older years should be read carefully, because they are based on fewer movies.

## Movies whose runtime is a statistical outlier

In [11]:
q1 = movies["runtime"].quantile(0.25)
q3 = movies["runtime"].quantile(0.75)
IQR = q3 - q1
lowerBound = q1 - (1.5*IQR)
upperBound = q3 + (1.5*IQR)
print("q1:",q1 , "q3:", q3, "IQR:", IQR, "lowerBound:", lowerBound, "upperBound:", upperBound )
print("Shorter than lower bound: ", len(movies[movies["runtime"] < lowerBound]))
print("Longer than upper bound: ", len(movies[(movies["runtime"]> upperBound)]))
runtime_outliers = movies[(movies["runtime"] < lowerBound) | (movies["runtime"]> upperBound)]
runtime_outliers[["title","runtime"]].sort_values(by="runtime", ascending=False)


q1: 99.0 q3: 126.0 IQR: 27.0 lowerBound: 58.5 upperBound: 166.5
Shorter than lower bound:  2
Longer than upper bound:  58


,title,runtime
324,Zack Snyder's Justice League,242
1227,Gone with the Wind,238
829,Once Upon a Time in America,229
1610,Lawrence of Arabia,228
1774,Ben-Hur,212
...,...,...
110,Saving Private Ryan,169
620,Gangs of New York,168
514,Dune: Part Two,167
2447,The Garden of Words,46


### Insight

1. IQR method was used to find runtime outliers. Q1 is 99 minutes and Q3 is 126 minutes, so the IQR is 27. The lower bound is 58.5 minutes and the upper bound is 166.5 minutes.

2. A movie is an outlier if its runtime is below the lower bound or above the upper bound. 60 movies are outliers: 2 are shorter than the lower bound and 58 are longer than the upper bound.

3. The longest movie is Zack Snyder's Justice League (242 minutes), followed by Gone with the Wind (238 minutes). The shortest are The Garden of Words (46 minutes) and The Guardians of the Galaxy Holiday Special (45 minutes).

4. Almost all outliers are long movies, so the unusual runtimes in this dataset are mostly very long films.

Note: the outliers were only listed and were not removed or changed, because the question asks to identify them.

Recommendation: Most movies run between about 99 and 126 minutes. Very long or very short films are rare, so they should be treated separately when analysing runtime.

## Primary genre and average popularity

In [12]:
primary = movie_genres.drop_duplicates(subset="movie_id",keep="first")
print("Finding whether primary genre alone is there: ", primary["movie_id"].is_unique)
print("movies with primary genres count: ", len(primary))
primary = pd.merge(primary,genres, on="genre_id", how="left")
movies_with_primary_genre = pd.merge(movies,primary[["movie_id","genre_name"]], on="movie_id",how="left")
print("movies without primary genre count: ", movies_with_primary_genre["genre_name"].isnull().sum())
movies_with_primary_genre.rename(columns={"genre_name":"primary_genre"},inplace=True)
movies_with_primary_genre[["title","primary_genre"]]


Finding whether primary genre alone is there:  True
movies with primary genres count:  2499
movies without primary genre count:  4


,title,primary_genre
0,Interstellar,Adventure
1,Inception,Action
2,The Avengers,Science Fiction
3,The Dark Knight,Action
4,Avatar,Action
...,...,...
2498,Daybreakers,Action
2499,Date Night,NaN
2500,Transformers,NaN
2501,The SpongeBob Movie: Sponge Out of Water,NaN


In [13]:
movies_with_primary_genre.groupby("primary_genre").popularity.agg(["mean","count"]).sort_values(by="mean",ascending=False)

,mean,count
primary_genre,,
Science Fiction,21.26,119
Music,18.32,10
Animation,16.81,139
Family,16.71,48
Action,16.39,474
War,15.47,23
Horror,14.85,223
Adventure,14.79,244
Fantasy,13.02,93


### Insight

1. A movie can have several genres, so movie_genres has more than one row per movie. The first row for each movie was kept as its primary genre. 

2. 2,499 movies have a primary genre. The other 4 movies have no row in movie_genres, so their primary genre is NaN. A left merge was used so these movies stay in the data.

3. Science Fiction has the highest average popularity (21.26, 119 movies), followed by Music (18.32), Animation (16.81), Family (16.71) and Action (16.39). Comedy and Romance are the lowest among the larger groups (9.98 each).

4. Some groups have very few movies (Documentary 1, Music 10, Western 10, History 12), so their averages are not reliable. Science Fiction, Animation and Action are based on many more movies, so they are more dependable.

Note: movie_genres has no order column, so "first genre" means the first row listed for each movie.

Recommendation: Science Fiction, Animation and Action films get the most attention on TMDB on average. Genres with very few movies should not be compared directly with the larger ones.

## Top 10 most prolific actors by number of movies, along with their average movie rating

In [14]:
cast_movies = pd.merge(cast,movies,on="movie_id",how="left")
print("movies without vote_average, count: ",cast_movies["vote_average"].isnull().sum())
cast_movies.groupby(["person_id","actor_name"]).agg(movie_count=("movie_id","nunique"),average_rating=("vote_average","mean")).sort_values(by="movie_count", ascending=False).head(10)

movies without vote_average, count:  0


,,movie_count,average_rating
person_id,actor_name,,
2231,Samuel L. Jackson,39,6.94
287,Brad Pitt,38,7.26
380,Robert De Niro,38,7.23
85,Johnny Depp,37,6.86
31,Tom Hanks,33,7.29
5293,Willem Dafoe,32,7.11
1245,Scarlett Johansson,32,7.06
13240,Mark Wahlberg,32,6.63
500,Tom Cruise,31,7.04


### Insight

1. The cast table has the actors but no ratings, and the movies table has the ratings but no actors. They were joined on movie_id with a left merge, so every cast row is kept. After the merge, vote_average has 0 missing values, so every cast row found its movie.

2. The data was grouped by person_id and actor_name, so two different actors with the same name are not merged. For each actor, the number of different movies and the average of vote_average were calculated. The result was sorted by movie count in descending order and the top 10 were kept.

3. Samuel L. Jackson has appeared in the most movies (39). Brad Pitt and Robert De Niro are tied for second place with 38 movies each, followed by Johnny Depp (37) and Tom Hanks (33). There is no tie at the 10th position, because the 11th actor has only 29 movies.

4. The average ratings of these actors are close to each other, from 6.63 (Mark Wahlberg) to 7.29 (Tom Hanks). Brad Pitt (7.26), Robert De Niro (7.23) and Tom Hanks (7.29) have the highest averages.

Note: the average rating is the average rating of the movies the actor appeared in. It does not measure the actor's own performance.

Recommendation: Actors with many movies are well-established names that studios can consider for casting. Their average rating should be read as the quality of the films they appeared in, not as a measure of their acting.

## Movie titles that appear more than once

In [15]:
movies_with_repeated_title = movies[movies["title"].duplicated(keep=False)]
print("count of movies, with movie title that appeared more than once: ", len(movies_with_repeated_title))
movies_with_repeated_title[["movie_id","title","year"]].sort_values(by="title")

count of movies, with movie title that appeared more than once:  100


,movie_id,title,year
878,377,A Nightmare on Elm Street,1984
1928,23437,A Nightmare on Elm Street,2010
261,812,Aladdin,1992
339,420817,Aladdin,2019
166,12155,Alice in Wonderland,2010
...,...,...,...
1737,60935,The Thing,2011
744,861,Total Recall,1990
860,64635,Total Recall,2012
252,1858,Transformers,2007


### Insight

1. 100 movies share their title with at least one other movie.

2. These 100 movies belong to 49 different titles. 47 titles appear twice (94 movies) and 2 titles appear three times (6 movies), so 47 x 2 + 2 x 3 = 100.

3. The list was sorted by title so repeated titles sit next to each other. Every repeated title has a different movie_id and a different release year. For example, A Nightmare on Elm Street (1984 and 2010), Aladdin (1992 and 2019), Total Recall (1990 and 2012) and Transformers (1987 and 2007).

Recommendation: Movies should be identified by movie_id and not by title, because the same title can belong to different films. Counting by title would merge them wrongly.

## Popularity vs vote count (unusual cases)

In [21]:
corr_value = movies["popularity"].corr(movies["vote_count"])
print("correlation (popularity Vs vote_count): ", round(corr_value,3))

correlation (popularity Vs vote_count):  0.361


In [17]:
pv = movies[["title","popularity","vote_count","year"]].copy()
pv["pop_rank"] = pv["popularity"].rank(pct=True)
pv["vote_rank"]= pv["vote_count"].rank(pct=True)
pv.head()

,title,popularity,vote_count,year,pop_rank,vote_rank
0,Interstellar,73.05,40566,2014,0.99,1.00
1,Inception,55.85,39727,2010,0.99,1.00
2,The Avengers,72.25,38993,2012,0.99,1.00
3,The Dark Knight,60.55,36255,2008,0.99,1.00
4,Avatar,48.20,34348,2009,0.98,1.00


In [18]:
high_pop_low_vote = pv[(pv["pop_rank"]>0.95) & (pv["vote_rank"]<0.60)]
high_vote_low_pop = pv[(pv["vote_rank"]>0.95) & (pv["pop_rank"]<0.60)]

print("High popularity, low vote_count: ", len(high_pop_low_vote))
print("High vote_count, low popularity: ", len(high_vote_low_pop))

High popularity, low vote_count:  28
High vote_count, low popularity:  1


In [19]:
high_pop_low_vote[["title","popularity","vote_count","year"]].sort_values(by="popularity", ascending=False)

,title,popularity,vote_count,year
2388,Disclosure Day,394.33,2328,2026
1315,Obsession,364.87,4203,2026
2248,Backrooms,258.47,2483,2026
1465,Michael,108.72,3846,2026
2452,Mortal Kombat II,99.52,2260,2026
1707,The Super Mario Galaxy Movie,97.12,3343,2026
1378,Avatar: Fire and Ash,95.53,4032,2025
2295,Lee Cronin's The Mummy,79.46,2430,2026
1776,Zootopia 2,63.11,3195,2025
1754,Captain America: Brave New World,60.36,3233,2025


In [20]:
high_vote_low_pop[["title","popularity","vote_count","year"]].sort_values(by="vote_count", ascending=False)

,title,popularity,vote_count,year
101,The Silence of the Lambs,7.06,18235,1991


### Insight

1. The correlation between popularity and vote_count is 0.361, a weak to moderate positive relationship. Movies with more votes tend to have higher popularity, but the link is loose, so many movies do not follow the pattern.

2. To compare both measures on the same scale, each movie was given a percentile rank (0 to 1) for popularity and for vote_count using rank(pct=True).

3. High popularity means a popularity rank above 0.95 with a vote_count rank below 0.60; high vote count means a vote_count rank above 0.95 with a popularity rank below 0.60.

4. 28 movies have high popularity but a low vote count. For example, Disclosure Day (popularity 394, 2,328 votes) and Obsession (popularity 365, 4,203 votes). Many of these movies are recent releases, which fits with popularity reflecting current attention while votes take time to build up.

5. Only 1 movie has a high vote count but low popularity: The Silence of the Lambs (popularity 7.06, 18,235 votes). It is an older film, so it has collected many votes over time but gets little current attention.

Note: the group sizes depend on the thresholds chosen, so different thresholds would give different groups.

Recommendation: Popularity alone can be misleading for new releases, because it can be high before enough votes are collected. Popularity should be read together with vote count.